# Music Annotation Project V4: Versioned GPU Training, Validation and GST Comparison

**Runtime:** T4/L4 GPU. Select **Runtime > Run all**.

This notebook selects an immutable dataset release, extracts its per-page TAR objects to Colab local storage, validates the assembled dataset, trains the configured YOLO segmentation model from scratch, evaluates validation data, runs GST normalisation comparisons, and copies durable outputs back to Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics

## Load configuration and perform the pre-training safety gate

The safety gate checks:

- the manifest and YAML exist;
- validation pages match the configuration;
- train and validation pages do not overlap;
- image and label filenames pair exactly;
- page prefixes in dataset images exactly match the manifest page lists;
- no validation page appears in the training image folder;
- a CUDA GPU is available.

If any check fails, training does not start.

In [ ]:
import cv2, json, hashlib, csv, re, shutil, tarfile
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO

CONFIG_PATH=Path('/content/drive/MyDrive/MusicAnnotationProject/project_config.json')
if not CONFIG_PATH.exists(): raise RuntimeError(f'Missing configuration file: {CONFIG_PATH}')
CONFIG=json.loads(CONFIG_PATH.read_text());ROOT=Path(CONFIG['project_root'])
DATASET_VERSIONS=ROOT/'dataset_versions';CURRENT_DATASET=DATASET_VERSIONS/'current_dataset.json'
REQUESTED_DATASET_VERSION=None  # e.g. 'v000017'; None selects current_dataset.json
GOLD=ROOT/'gold_standard_testing';RUNS=ROOT/'training_runs';PREDS=ROOT/'predictions';ALIGNED=ROOT/'aligned'
T=CONFIG['training'];FIXED_VALIDATION=set(CONFIG['validation_pages']);EXTENSIONS={'.jpg','.jpeg','.png','.tif','.tiff'}
STAGE_DATASET_LOCALLY=True
LOCAL_ROOT=Path('/content/MusicAnnotationProject_local');LOCAL_DATASET=LOCAL_ROOT/'dataset';LOCAL_YAML=LOCAL_ROOT/'dataset_v4_local.yaml';LOCAL_RUNS=LOCAL_ROOT/'training_runs';LOCAL_PREDS=LOCAL_ROOT/'predictions';LOCAL_ALIGNED=LOCAL_ROOT/'aligned';LOCAL_GOLD=LOCAL_ROOT/'gold_standard_testing';LOCAL_MANIFEST=LOCAL_ROOT/'dataset_manifest_v3.json';LOCAL_RELEASE=LOCAL_ROOT/'dataset_release.json'
DRIVE_GOLD=GOLD;DRIVE_RUNS=RUNS;DRIVE_PREDS=PREDS;DRIVE_ALIGNED=ALIGNED
RUN_GST_NORMALISATION_COMPARISON=True;BACKGROUND_BLUR_SIGMA=75;CLAHE_CLIP_LIMIT=2.0;CLAHE_GRID_SIZE=(8,8)
GST_TILE_SIZE=int(CONFIG['gold_standard']['tile_size']);GST_TILE_STRIDE=int(CONFIG['gold_standard']['tile_stride']);GST_CONFIDENCE=float(T['prediction_confidence']);GST_IMAGE_SIZE=int(T['image_size'])

def images(folder):
    folder=Path(folder);return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS) if folder.exists() else []
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
def page_prefix_from_tile(stem):
    match=re.match(r'^(.*)_y\d+_x\d+$',stem);return match.group(1) if match else None
def load_json(path,label):
    path=Path(path)
    if not path.exists(): raise RuntimeError(f'Missing {label}: {path}')
    return json.loads(path.read_text())

if REQUESTED_DATASET_VERSION is None:
    pointer=load_json(CURRENT_DATASET,'current dataset pointer');dataset_version_name=pointer['version_name'];release_path=ROOT/pointer['release_file']
else:
    dataset_version_name=REQUESTED_DATASET_VERSION;release_path=DATASET_VERSIONS/f'{dataset_version_name}.json'
dataset_release=load_json(release_path,'dataset release manifest')
if dataset_release.get('version_name')!=dataset_version_name: raise RuntimeError('Dataset pointer/version mismatch')
manifest=dataset_release['manifest_snapshot'];released_config=dataset_release['config_snapshot'];T=released_config['training'];FIXED_VALIDATION=set(released_config['validation_pages'])
train_pages=set(manifest['training_pages']);val_pages=set(manifest['validation_pages'])
if FIXED_VALIDATION-val_pages: raise RuntimeError(f'Fixed validation pages missing: {sorted(FIXED_VALIDATION-val_pages)}')
if train_pages & val_pages: raise RuntimeError(f'Train/validation overlap: {sorted(train_pages & val_pages)}')
release_train={x['page'] for x in dataset_release['page_objects'] if x['split']=='train'};release_val={x['page'] for x in dataset_release['page_objects'] if x['split']=='val'}
if release_train!=train_pages or release_val!=val_pages: raise RuntimeError('Release page lists do not match manifest')
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime > Change runtime type > GPU, then run all again.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+dataset_version_name+'_'+Path(T['model']).stem
RUN_DIR=RUNS/RUN;RUN_PRED=PREDS/RUN
print('GPU:',torch.cuda.get_device_name(0));print('Dataset:',dataset_version_name,dataset_release['release_hash']);print('Run:',RUN);print('Training tiles:',dataset_release['training_tile_count']);print('Validation tiles:',dataset_release['validation_tile_count']);print('PRE-STAGING SAFETY GATE PASSED')


## Assemble the selected dataset release on the Colab local server

Each page object is one uncompressed TAR file. The trainer extracts the exact objects listed by the selected immutable release. No growing monolithic ZIP is built or copied.


In [ ]:
def copy_tree_contents(source,destination):
    source=Path(source);destination=Path(destination)
    if not source.exists(): raise RuntimeError(f'Required source missing: {source}')
    for path in source.rglob('*'):
        if path.is_file() and not path.name.endswith('.cache'):
            target=destination/path.relative_to(source);target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(path,target)

if STAGE_DATASET_LOCALLY:
    if LOCAL_ROOT.exists(): shutil.rmtree(LOCAL_ROOT)
    LOCAL_ROOT.mkdir(parents=True);LOCAL_RUNS.mkdir();LOCAL_PREDS.mkdir();LOCAL_ALIGNED.mkdir();LOCAL_GOLD.mkdir()
    for index,item in enumerate(dataset_release['page_objects'],1):
        object_path=ROOT/item['relative_path']
        if not object_path.exists(): raise RuntimeError(f'Missing page object: {object_path}')
        if object_path.stem!=item['object_hash']: raise RuntimeError(f'Object name/hash mismatch: {object_path}')
        with tarfile.open(object_path,'r') as archive:
            members=[m for m in archive.getmembers() if m.isfile()]
            if len(members)!=int(item['tile_count'])*3: raise RuntimeError(f'{item["page"]}: unexpected TAR member count')
            archive.extractall(LOCAL_ROOT,filter='data')
        if index%25==0 or index==len(dataset_release['page_objects']): print('Extracted pages:',index,'/',len(dataset_release['page_objects']))
    copy_tree_contents(DRIVE_GOLD,LOCAL_GOLD);copy_tree_contents(DRIVE_ALIGNED,LOCAL_ALIGNED)
    LOCAL_MANIFEST.write_text(json.dumps(manifest,indent=2));LOCAL_RELEASE.write_text(json.dumps(dataset_release,indent=2))
    LOCAL_YAML.write_text(f'path: {LOCAL_DATASET}\ntrain: images/train\nval: images/val\nnames:\n  0: annotation\n')
    DATASET=LOCAL_DATASET;YAML=LOCAL_YAML;MANIFEST=LOCAL_MANIFEST;GOLD=LOCAL_GOLD;ALIGNED=LOCAL_ALIGNED;RUNS=LOCAL_RUNS;PREDS=LOCAL_PREDS;RUN_DIR=RUNS/RUN;RUN_PRED=PREDS/RUN

folder_page_prefixes={}
for split,expected_pages,expected_count in [('train',train_pages,int(dataset_release['training_tile_count'])),('val',val_pages,int(dataset_release['validation_tile_count']))]:
    image_map={p.stem:p for p in images(DATASET/'images'/split)};label_map={p.stem:p for p in (DATASET/'labels'/split).glob('*.txt')};mask_map={p.stem:p for p in images(DATASET/'mask_tiles'/split)}
    if set(image_map)!=set(label_map) or set(image_map)!=set(mask_map): raise RuntimeError(f'{split}: local image/label/mask mismatch')
    if len(image_map)!=expected_count: raise RuntimeError(f'{split}: expected {expected_count} tiles, found {len(image_map)}')
    bad=[s for s in image_map if page_prefix_from_tile(s) is None]
    if bad: raise RuntimeError(f'{split}: invalid tile names: {bad[:20]}')
    prefixes={page_prefix_from_tile(s) for s in image_map};folder_page_prefixes[split]=prefixes
    if prefixes!=expected_pages: raise RuntimeError(f'{split}: release/manifest page mismatch')
    print(f'{split}: {len(image_map)} image/label/mask sets across {len(prefixes)} pages')
if folder_page_prefixes['train'] & folder_page_prefixes['val']: raise RuntimeError('Page prefixes occur in both splits')
print('LOCAL VERSIONED STAGING PASSED:',dataset_version_name)


## Train YOLO and evaluate quantitative validation pages

Early stopping is expected when validation fitness does not improve for the configured patience period. Evaluation uses `best.pt`.

In [ ]:
model=YOLO(T['model'])
model.train(
    data=str(YAML),
    epochs=T['epochs'],
    patience=T['patience'],
    imgsz=T['image_size'],
    batch=T['batch_size'],
    device=0,
    workers=2,
    project=str(RUNS),
    name=RUN,
    exist_ok=False,
    seed=T['seed'],
    plots=True,
    cache='disk',
)

BEST=RUN_DIR/'weights'/'best.pt'
if not BEST.exists():
    raise RuntimeError('best.pt missing after training.')

trained=YOLO(str(BEST))
metrics=trained.val(
    data=str(YAML),
    split='val',
    imgsz=T['image_size'],
    device=0,
    project=str(RUN_PRED),
    name='metrics',
    exist_ok=True,
)
metric_dict={key:float(value) for key,value in metrics.results_dict.items()}

trained.predict(
    source=str(DATASET/'images'/'val'),
    save=True,
    conf=T['prediction_confidence'],
    imgsz=T['image_size'],
    device=0,
    project=str(RUN_PRED),
    name='validation_tiles',
    exist_ok=True,
    retina_masks=True,
)

for page in sorted(val_pages):
    hits=[p for p in images(ALIGNED) if p.stem.lower()==page.lower()]
    if not hits:
        print('Missing aligned validation page:',page)
        continue
    trained.predict(
        source=str(hits[0]),
        save=True,
        conf=T['prediction_confidence'],
        imgsz=1280,
        device=0,
        project=str(RUN_PRED),
        name='validation_full_pages',
        exist_ok=True,
        retina_masks=True,
    )

print(json.dumps(metric_dict,indent=2))

## GST original/background-normalised/CLAHE comparison

Each GST page is evaluated three ways using the same trained model and overlapping tiles. No binary threshold is applied to the input page because faint pencil could be removed.

In [ ]:
GST_COMPARISON_ROOT=RUN_PRED/'gst_normalisation_comparison'
GST_NORMALISED_DIR=GST_COMPARISON_ROOT/'normalised_pages'
GST_PREDICTION_DIR=GST_COMPARISON_ROOT/'predictions'
GST_COMPARISON_DIR=GST_COMPARISON_ROOT/'comparisons'
for folder in [GST_COMPARISON_ROOT,GST_NORMALISED_DIR,GST_PREDICTION_DIR,GST_COMPARISON_DIR]:
    folder.mkdir(parents=True,exist_ok=True)


def normalise_background(image,blur_sigma=BACKGROUND_BLUR_SIGMA):
    background=cv2.GaussianBlur(image,(0,0),blur_sigma)
    background=np.maximum(background,1)
    return cv2.divide(image,background,scale=255)


def apply_clahe(image):
    clahe=cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT,tileGridSize=CLAHE_GRID_SIZE)
    return clahe.apply(image)


def gst_tile_positions(length):
    if length<=GST_TILE_SIZE:
        return [0]
    positions=list(range(0,length-GST_TILE_SIZE+1,GST_TILE_STRIDE))
    final_position=length-GST_TILE_SIZE
    if positions[-1]!=final_position:
        positions.append(final_position)
    return positions


def tiled_gst_predict(gray_image):
    colour_image=cv2.cvtColor(gray_image,cv2.COLOR_GRAY2BGR)
    height,width=gray_image.shape
    padded_height=max(height,GST_TILE_SIZE)
    padded_width=max(width,GST_TILE_SIZE)
    canvas=np.full((padded_height,padded_width,3),255,np.uint8)
    canvas[:height,:width]=colour_image
    confidence_map=np.zeros((padded_height,padded_width),np.float32)
    instance_records=[]

    for y in gst_tile_positions(padded_height):
        for x in gst_tile_positions(padded_width):
            tile=canvas[y:y+GST_TILE_SIZE,x:x+GST_TILE_SIZE]
            result=trained.predict(
                tile,
                conf=GST_CONFIDENCE,
                imgsz=GST_IMAGE_SIZE,
                device=0,
                verbose=False,
                retina_masks=True,
            )[0]
            if result.masks is None:
                continue
            masks=result.masks.data.cpu().numpy()
            confidences=result.boxes.conf.cpu().numpy() if result.boxes is not None else np.ones(len(masks))
            for mask,confidence in zip(masks,confidences):
                mask=cv2.resize(mask,(GST_TILE_SIZE,GST_TILE_SIZE),interpolation=cv2.INTER_NEAREST)>0.5
                mask_pixels=int(np.sum(mask))
                tile_coverage_pct=100*mask_pixels/(GST_TILE_SIZE*GST_TILE_SIZE)
                instance_records.append({
                    'tile_x':int(x),
                    'tile_y':int(y),
                    'confidence':float(confidence),
                    'mask_pixels':mask_pixels,
                    'tile_coverage_pct':tile_coverage_pct,
                })
                region=confidence_map[y:y+GST_TILE_SIZE,x:x+GST_TILE_SIZE]
                update=mask & (confidence>region)
                region[update]=confidence

    confidence_map=confidence_map[:height,:width]
    selected=confidence_map>=GST_CONFIDENCE
    highlighted=colour_image.copy()
    highlighted[selected]=(0,0,255)
    prediction=cv2.addWeighted(colour_image,.68,highlighted,.32,0)
    return prediction,confidence_map,instance_records


gst_results=[]
gst_instance_results=[]

if RUN_GST_NORMALISATION_COMPARISON:
    gst_files=images(GOLD)
    print('GST files:',len(gst_files))
    for source_path in gst_files:
        original=cv2.imread(str(source_path),cv2.IMREAD_GRAYSCALE)
        if original is None:
            print('Unreadable GST file:',source_path)
            continue
        background_normalised=normalise_background(original)
        clahe_normalised=apply_clahe(background_normalised)
        versions={'original':original,'bgnorm':background_normalised,'clahe':clahe_normalised}
        page_normalised_dir=GST_NORMALISED_DIR/source_path.stem
        page_prediction_dir=GST_PREDICTION_DIR/source_path.stem
        page_normalised_dir.mkdir(parents=True,exist_ok=True)
        page_prediction_dir.mkdir(parents=True,exist_ok=True)
        version_outputs={}

        for version_name,page_image in versions.items():
            normalised_path=page_normalised_dir/f'{source_path.stem}_{version_name}.png'
            cv2.imwrite(str(normalised_path),page_image)
            prediction,confidence_map,instance_records=tiled_gst_predict(page_image)
            prediction_path=page_prediction_dir/f'{source_path.stem}_{version_name}_prediction.png'
            cv2.imwrite(str(prediction_path),prediction)
            version_outputs[version_name]=prediction
            selected=confidence_map>=GST_CONFIDENCE
            predicted_pixels=int(np.sum(selected))
            page_pixels=int(page_image.size)
            predicted_page_pct=100*predicted_pixels/page_pixels
            selected_confidences=confidence_map[selected]
            instance_confidences=[record['confidence'] for record in instance_records]
            result_record={
                'file':source_path.name,
                'version':version_name,
                'instances':len(instance_records),
                'predicted_pixels':predicted_pixels,
                'page_pixels':page_pixels,
                'predicted_page_pct':predicted_page_pct,
                'mean_pixel_confidence':float(selected_confidences.mean()) if predicted_pixels else 0.0,
                'maximum_confidence':float(confidence_map.max()),
                'median_instance_confidence':float(np.median(instance_confidences)) if instance_confidences else 0.0,
                'p95_instance_confidence':float(np.percentile(instance_confidences,95)) if instance_confidences else 0.0,
                'largest_instance_tile_pct':max((record['tile_coverage_pct'] for record in instance_records),default=0.0),
                'prediction_file':str(prediction_path),
            }
            gst_results.append(result_record)
            for record in instance_records:
                gst_instance_results.append({'file':source_path.name,'version':version_name,**record})
            print(
                source_path.name,version_name,
                'instances:',len(instance_records),
                'page %:',round(predicted_page_pct,3),
                'max confidence:',round(result_record['maximum_confidence'],4),
                'largest tile mask %:',round(result_record['largest_instance_tile_pct'],3),
            )

        comparison=np.hstack([version_outputs['original'],version_outputs['bgnorm'],version_outputs['clahe']])
        cv2.imwrite(str(GST_COMPARISON_DIR/f'{source_path.stem}_comparison.png'),comparison)
else:
    print('GST normalisation comparison disabled.')

## Display GST normalisation comparisons

Each displayed image is ordered:

1. original-input prediction;
2. background-normalised prediction;
3. background-normalised plus CLAHE prediction.

In [ ]:
if RUN_GST_NORMALISATION_COMPARISON:
    comparison_files=sorted(GST_COMPARISON_DIR.glob('*_comparison.png'))
    for path in comparison_files:
        image=cv2.imread(str(path))
        plt.figure(figsize=(22,12))
        plt.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB))
        plt.title(path.name+' | original | background normalised | CLAHE')
        plt.axis('off')
        plt.show()

## Export GST diagnostics and combined run summary

In [ ]:
GST_SUMMARY_JSON=GST_COMPARISON_ROOT/'normalisation_summary.json'
GST_SUMMARY_CSV=GST_COMPARISON_ROOT/'normalisation_summary.csv'
GST_INSTANCE_CSV=GST_COMPARISON_ROOT/'gst_instance_predictions.csv'

if RUN_GST_NORMALISATION_COMPARISON:
    GST_SUMMARY_JSON.write_text(json.dumps({
        'model':str(BEST),
        'confidence':GST_CONFIDENCE,
        'tile_size':GST_TILE_SIZE,
        'tile_stride':GST_TILE_STRIDE,
        'background_blur_sigma':BACKGROUND_BLUR_SIGMA,
        'clahe_clip_limit':CLAHE_CLIP_LIMIT,
        'results':gst_results,
    },indent=2))

    with open(GST_SUMMARY_CSV,'w',newline='',encoding='utf-8') as csv_file:
        if gst_results:
            writer=csv.DictWriter(csv_file,fieldnames=list(gst_results[0].keys()))
            writer.writeheader(); writer.writerows(gst_results)

    with open(GST_INSTANCE_CSV,'w',newline='',encoding='utf-8') as csv_file:
        if gst_instance_results:
            writer=csv.DictWriter(csv_file,fieldnames=list(gst_instance_results[0].keys()))
            writer.writeheader(); writer.writerows(gst_instance_results)

summary={
    'run_name':RUN,
    'dataset_version':dataset_version_name,
    'dataset_release_hash':dataset_release['release_hash'],
    'dataset_release_file':str(release_path),
    'model':T['model'],
    'training_pages':sorted(train_pages),
    'validation_pages':sorted(val_pages),
    'training_page_count':len(train_pages),
    'validation_page_count':len(val_pages),
    'training_tile_count':len(images(DATASET/'images'/'train')),
    'validation_tile_count':len(images(DATASET/'images'/'val')),
    'metrics':metric_dict,
    'manifest_md5':md5(MANIFEST),
    'best_model':str(BEST),
    'gst_normalisation_enabled':RUN_GST_NORMALISATION_COMPARISON,
    'gst_results':gst_results,
    'gst_instance_csv':str(GST_INSTANCE_CSV) if RUN_GST_NORMALISATION_COMPARISON else None,
}

shutil.copy2(LOCAL_RELEASE,RUN_DIR/'dataset_release.json')
(RUN_DIR/'run_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('Run summary:',RUN_DIR/'run_summary.json')
if RUN_GST_NORMALISATION_COMPARISON:
    print('GST comparisons:',GST_COMPARISON_DIR)
    print('GST page summary:',GST_SUMMARY_CSV)
    print('GST instance diagnostics:',GST_INSTANCE_CSV)

## Copy completed outputs back to Google Drive

Colab local storage is temporary. This final step copies the complete local training run and prediction run to their durable Google Drive folders.

In [ ]:
if STAGE_DATASET_LOCALLY:
    drive_run_dir=DRIVE_RUNS/RUN
    drive_prediction_dir=DRIVE_PREDS/RUN
    if drive_run_dir.exists(): shutil.rmtree(drive_run_dir)
    if drive_prediction_dir.exists(): shutil.rmtree(drive_prediction_dir)
    shutil.copytree(RUN_DIR,drive_run_dir)
    shutil.copytree(RUN_PRED,drive_prediction_dir)
    print('Training run copied to:',drive_run_dir)
    print('Predictions copied to:',drive_prediction_dir)
    print('Best model copied to:',drive_run_dir/'weights'/'best.pt')